# Week 3 — Recidivism through Predictive Models

* * *

<div class="alert alert-success">  
    
### Learning Objectives 
    
* Build a small logistic-regression and decision-tree model on the COMPAS dataset using `scikit-learn`.
* Understand what a model's accuracy, false-positive rate, and false-negative rate are — and why "high accuracy overall" can hide unequal harm.
* See how a model that never receives `race` as a feature can still produce racially patterned errors via *proxy variables*.
* Encounter the fairness-impossibility result: equalizing different fairness metrics across groups is mathematically impossible when base rates differ.
* Connect O'Neil's *Weapons of Math Destruction* and Dressel & Farid's accuracy critique to a model you built yourself.
</div>

### Icons Used in This Notebook
🔔 **Question**: A quick question to help you understand what's going on.<br>
💡 **Tip**: How to do something a bit more efficiently or effectively.<br>
⚠️ **Warning:** Heads-up about tricky stuff or common mistakes.<br>
💭 **Reflection**: Reflecting on ethical implications, biases, and social impact in data science.

### Sections
1. [Framing: O'Neil's Weapons of Math Destruction](#wmd)
2. [What Is Logistic Regression, Really?](#logreg)
3. [Loading and Preparing COMPAS](#load)
4. [Feature Selection and the Politics of Proxies](#features)
5. [Train, Predict, and Score](#train)
6. [Looking at the Coefficients](#coefs)
7. [Confusion Matrix — Overall and By Race](#cm)
8. [A Decision Tree for Comparison](#tree)
9. [Dressel & Farid: Is COMPAS Even Accurate?](#dressel)
10. [The Fairness-Impossibility Result](#impossibility)
11. [Reflection Prompts](#reflection)

<a id='wmd'></a>
# 1. Framing: O'Neil's Weapons of Math Destruction

Monday's reading is Chapter 5 of Cathy O'Neil's *Weapons of Math Destruction* (2016). She defines a **WMD** by three properties:

- **Opaque** — The people affected by the model can't see how it works, or even *that* it's making a decision about them.
- **Scalable** — One model makes decisions for thousands or millions of people. Errors don't average out; they multiply.
- **Damaging** — The decisions matter materially: bail, hiring, housing, sentencing.

Recidivism scoring is one of her central case studies. This week we *build* one of these scores ourselves, deliberately. The point of doing so is not to celebrate the technique — it's to feel the gap between how easy it is to write the code and how high the stakes are.

> **Data transparency note**: We're about to fit a logistic regression on the same COMPAS data we explored in Week 2. We are, in a small way, *reproducing* the kind of system the readings critique. That tension is intentional. Building a thing makes its assumptions visible in a way that reading critiques alone does not. Treat the model below not as a system you'd deploy, but as a teaching artifact — an exhibit, not a tool.

<a id='logreg'></a>
# 2. What Is Logistic Regression, Really?

For beginners — no calculus required.

A **logistic regression model** takes some inputs (let's call them *features*) and outputs a single number between 0 and 1. We interpret that number as the model's predicted *probability* that the answer is "yes" (e.g., "will reoffend within 2 years"). To turn the probability into a hard yes/no, we pick a *threshold* — usually 0.5.

The model has one number per feature, called a **coefficient** or **weight**. Big positive coefficients mean "more of this feature pushes the prediction toward yes." Big negative coefficients mean the opposite.

That's it. Underneath the math, that's the whole shape of the model.

In [ ]:
# Tiny toy example to build intuition
import numpy as np
from sklearn.linear_model import LogisticRegression

# Feature: hours studied. Outcome: passed (1) or failed (0).
X_toy = np.array([[1], [2], [3], [4], [5], [6], [7], [8]])
y_toy = np.array([0, 0, 0, 0, 1, 1, 1, 1])

toy_model = LogisticRegression().fit(X_toy, y_toy)
for hrs in [1, 3, 5, 7]:
    p = toy_model.predict_proba([[hrs]])[0, 1]
    print(f"{hrs} hours studied → predicted P(pass) = {p:.2f}")

💡 **Tip**: That's it for the math intuition. Now we apply the same technique to a much higher-stakes problem. The mechanics will look the same. The ethics will not.

<a id='load'></a>
# 3. Loading and Preparing COMPAS

This builds on Week 2 — we'll re-do the cleaning quickly.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

url = "https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv"
df = pd.read_csv(url)

# Cleaning (same as Week 2)
df = df[df["race"].isin(["African-American", "Caucasian"])].copy()
df["race"] = df["race"].replace({"African-American": "Black", "Caucasian": "White"})
df = df.dropna(subset=["two_year_recid", "priors_count", "age", "c_charge_degree", "sex"])
df = df[(df["days_b_screening_arrest"] <= 30) & (df["days_b_screening_arrest"] >= -30)]
print("Cleaned shape:", df.shape)
df[["age", "priors_count", "c_charge_degree", "sex", "race", "two_year_recid"]].head()

<a id='features'></a>
# 4. Feature Selection and the Politics of Proxies

We'll use four features as inputs:

- `age` (numeric)
- `priors_count` (numeric — number of prior offenses)
- `c_charge_degree` (categorical — F = felony, M = misdemeanor)
- `sex` (categorical — Male / Female)

We will **deliberately not use `race`** as a feature.

Does this make the model race-neutral?

**No.** And that's the most important lesson of this week. Several of our "neutral" features are themselves *proxies* for race in the United States:

- `priors_count` reflects past arrests, which reflect police presence, which is uneven by neighborhood and race.
- `c_charge_degree` reflects prosecutorial decisions — the same act can be charged as a felony or misdemeanor depending on the discretion of the prosecutor.

When a model is fit on data shaped by structural racism, race-blindness in the *features* doesn't produce race-blindness in the *predictions*. It just hides the path the bias takes.

In [ ]:
# Set up features and target
features_df = df[["age", "priors_count", "c_charge_degree", "sex"]]
y = df["two_year_recid"]
race = df["race"]   # we'll keep race aside, only to evaluate the model later

In [ ]:
# One-hot encode the categoricals.
# This turns 'c_charge_degree' (with values F, M) into two 0/1 columns.
X = pd.get_dummies(features_df, columns=["c_charge_degree", "sex"], drop_first=True).astype(float)
X.head()

💡 **Tip**: `drop_first=True` drops one category from each variable to avoid redundancy (because if `sex_Male` is 0, the person must be female, so we don't need a separate `sex_Female` column). This is called avoiding the "dummy variable trap."

<a id='train'></a>
# 5. Train, Predict, and Score

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# Split data: 80% for training, 20% for testing
# stratify=y keeps the same recidivism rate in train and test sets
X_train, X_test, y_train, y_test, race_train, race_test = train_test_split(
    X, y, race, test_size=0.2, random_state=42, stratify=y)

print("Train size:", X_train.shape, "| Test size:", X_test.shape)

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

print(f"Training accuracy: {model.score(X_train, y_train):.3f}")
print(f"Test accuracy:     {model.score(X_test, y_test):.3f}")

💭 **Reflection**: The test accuracy is around **0.65** — better than chance, far from infallible. Before reading anything more into that number: what does "accuracy" actually measure here? It tells us the fraction of test-set cases where the predicted yes/no matched the actual yes/no. It tells us *nothing* about whether the errors are evenly distributed across groups, or whether the people the model misclassifies are the ones who can least afford to be misclassified.

<a id='coefs'></a>
# 6. Looking at the Coefficients

What did the model decide *was* important?

In [ ]:
coefs = pd.Series(model.coef_[0], index=X.columns).sort_values(ascending=False)
coefs

🔔 **Question**: Look at the signs and magnitudes. You'll see large positive coefficients on `sex_Male` and `priors_count`: being male and having more priors both push the prediction toward "will reoffend." `age` is mildly negative (older → lower predicted recidivism), and `c_charge_degree_M` (misdemeanor) is negative compared to felonies.

⚠️ A tempting trap: the *raw* coefficient on `sex_Male` (~0.38) looks larger than `priors_count` (~0.17), so you might conclude the model "cares more about sex." But coefficients aren't directly comparable across features on different scales — `sex_Male` is 0/1, while `priors_count` ranges from 0 to 30+. In practice, having many priors moves the prediction much more than being male does.

More important than the *math* of the coefficients is their *meaning*. `priors_count` is a record of past police contact, not of underlying criminality. When the model uses past police contact to predict future police contact, what is it actually predicting?

<a id='cm'></a>
# 7. Confusion Matrix — Overall and By Race

A **confusion matrix** breaks predictions into four buckets: true positive, false positive, true negative, false negative. The off-diagonal cells are the errors.

We'll first compute it overall, then disaggregate by race.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred = model.predict(X_test)
print("Overall confusion matrix:")
print(pd.DataFrame(confusion_matrix(y_test, y_pred),
                   index=["actual: no recid", "actual: recid"],
                   columns=["pred: no recid", "pred: recid"]))
print()
print(classification_report(y_test, y_pred, target_names=["no recid", "recid"]))

In [ ]:
# Now broken out by race.
def rates_by_group(y_true, y_pred, group):
    out = []
    for g in sorted(group.unique()):
        mask = group == g
        cm = confusion_matrix(y_true[mask], y_pred[mask])
        tn, fp, fn, tp = cm.ravel()
        fpr = fp / (fp + tn) if (fp + tn) else float('nan')   # false positive rate
        fnr = fn / (fn + tp) if (fn + tp) else float('nan')   # false negative rate
        out.append({"group": g, "n": mask.sum(), "FPR": round(fpr, 3), "FNR": round(fnr, 3)})
    return pd.DataFrame(out)

rates_by_group(y_test.reset_index(drop=True),
               pd.Series(y_pred),
               race_test.reset_index(drop=True))

💭 **Reflection**: The model never saw `race` as a feature. And yet the FPR for Black defendants in the test set is roughly **2× higher** than for White defendants (about 0.36 vs 0.19), and the FNR for White defendants is correspondingly higher than for Black defendants (about 0.58 vs 0.34). The pattern from Week 2 reproduces *itself*, almost identically, in a model we built without ever mentioning race.

This is the proxy-variable problem made concrete. "Race-blind" is not the same as "race-neutral." The model learned the disparity through the back door — through correlations between the features it *could* see (`priors_count`, `c_charge_degree`) and the racial position it *couldn't*.

<a id='tree'></a>
# 8. A Decision Tree for Comparison

Let's swap out the model entirely. A **decision tree** asks a sequence of yes/no questions and routes you to a final prediction. It's a fundamentally different shape of model — much simpler, much more interpretable. Will it produce different results?

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train, y_train)
print(f"Tree test accuracy: {tree.score(X_test, y_test):.3f}")

y_pred_tree = tree.predict(X_test)
rates_by_group(y_test.reset_index(drop=True),
               pd.Series(y_pred_tree),
               race_test.reset_index(drop=True))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
plot_tree(tree, feature_names=list(X.columns), class_names=["no recid", "recid"],
          filled=True, rounded=True, ax=ax)
plt.tight_layout()
plt.show()

🔔 **Question**: Different model, similar disparity. What does this tell you about whether the bias lives in the model, or in the data?

<a id='dressel'></a>
# 9. Dressel & Farid: Is COMPAS Even Accurate?

A 2018 paper by Julia Dressel and Hany Farid added another wrinkle. They asked: *how accurate is COMPAS, really?* They recruited untrained Mechanical Turk workers, gave them just **seven** features about each defendant, and asked them to predict re-arrest.

The result: untrained crowd workers matched COMPAS's accuracy. A simple two-feature linear model (age + priors) also matched COMPAS. The proprietary, 137-question instrument that judges and parole boards have trusted for years was not detectably better than a layperson reading a paragraph.

If COMPAS is no more accurate than untrained humans answering 7 questions, why do we trust it? 

O'Neil's answer is the "scalable" criterion. It's not that the algorithm is *better*. It's that it's *automated, fast, cheap, and feels objective*. That combination is what makes it worth deploying — to the deployer. Not to the people it predicts about.

<a id='impossibility'></a>
# 10. The Fairness-Impossibility Result

Northpointe defended COMPAS by pointing to a *different* fairness metric than ProPublica did. They argued COMPAS was "calibrated" — meaning a high-risk score had the same predictive value (i.e., similar actual recidivism rate) for both Black and White defendants. ProPublica responded by pointing to the unequal error rates we just reproduced.

Who was right?

In 2017, the statistician Alexandra Chouldechova proved a startling result: **when base rates differ between two groups, you cannot simultaneously equalize calibration AND equalize false-positive and false-negative rates across the groups.** It's mathematically impossible. (Kleinberg, Mullainathan, and Raghavan proved a related result independently.)

In other words: as long as the *real* recidivism rates differ between the two populations (and they do, because the data we're using *measures* re-arrest, which is shaped by uneven policing), no model — fancy or simple, transparent or opaque — can satisfy *both* fairness criteria at once.

**Which definition of fairness you choose is therefore not a math question. It is a values question.**

⚠️ **Warning**: This impossibility result is sometimes used to throw up our hands: "there's no such thing as a fair algorithm, so any algorithm is fine." That's not the lesson. The lesson is that designing a high-stakes algorithmic system requires *committing*, openly, to a particular notion of fairness — and being accountable to the harms that follow from that choice. Pretending the choice doesn't exist is itself a choice.

💭 **Reflection**: If you had to choose between equalizing calibration and equalizing FPR/FNR for a system that decided pretrial detention, which would you pick? Why? What would you say to the people harmed by the metric you *didn't* equalize?

### Other tools you could use for fairness analysis

`scikit-learn` is one entry point; the broader ecosystem of fairness tooling is worth knowing about. We'll mention these briefly in class:

- **Fairlearn** (Microsoft) — a Python library for assessing and mitigating fairness issues in ML, with a friendly API for group-disaggregated metrics.
- **Aequitas** (University of Chicago) — a fairness audit toolkit aimed at policy-facing analyses; produces clean reports.
- **AIF360** (IBM) — a richer fairness toolkit with multiple debiasing algorithms; powerful but heavier to learn.
- **R** (`caret`, `tidymodels`) — equivalent ML pipelines in R, popular in academic statistics.
- **Weka** — a GUI-based ML tool helpful for getting intuition without writing code.

If your final project involves fairness audits, Fairlearn is a great next step beyond the `confusion_matrix`-by-group pattern we used here.

<a id='reflection'></a>
# 11. Reflection Prompts

For your discussion post on recidivism and predictive models, you can start from any of:

1. We built a model without using race as a feature, and it still produced racially patterned errors. What does this tell you about the limits of "race-blindness" as a goal in algorithmic design?

2. O'Neil's WMD criteria — opaque, scalable, damaging — describe COMPAS well. Pick another decision-making algorithm in your own life (a credit-card fraud detector, a college admissions metric, a hiring screen, a content recommender) and apply the WMD test.

3. Dressel & Farid showed that COMPAS is no better than untrained humans answering 7 questions. Why do you think it remains in use? Whose interests does its continued deployment serve?

4. Apply the fairness-impossibility result to a domain you care about. What would committing publicly to one fairness criterion *look like*, in concrete policy terms?

<div class="alert alert-success">

## ❗ Key Points

* Logistic regression and decision trees are simple, accessible ML techniques — and that's the point. Building a small recidivism predictor takes a few lines of `scikit-learn`. The ethics are much harder than the code.
* A model that doesn't use `race` as a feature can still produce racially patterned errors via *proxy variables* (e.g., `priors_count`). "Race-blindness" ≠ "race-neutrality."
* Different model families (logistic regression, decision tree) give similar disparities on the same data — meaning the bias lives in the data and the world that produced it, not in the choice of algorithm.
* Dressel & Farid show that COMPAS is no more accurate than untrained humans answering seven questions. Its persistence is about scale and the appearance of objectivity, not about technical superiority.
* Chouldechova and Kleinberg et al. proved that you cannot equalize calibration and error rates simultaneously when base rates differ. The choice between fairness definitions is a values question, not a math question.

</div>